# LatencyOptimization

**Module 13 · Lesson 13.4**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Latency Has Two Numbers, And Users Feel TTFT
- Stream The Output
- Prefill vs Decode: Where The Time Goes
- Cut TTFT: Prompt Caching, A Shorter Prompt, A Faster Model
- Cut Decode: Shorter Output And Speculative Decoding
- Do Less Waiting: Parallel Calls And Semantic Caching

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic -q

# Reproducibility - the lesson uses seeded random values throughout

## Accurate, cheap, and too slow to keep the user

> 💡 **Info**
>
> Your assistant answers well and, after Lessons 13.1 through 13.3, it answers cheaply. But it takes **several seconds** to respond, and a real fraction of users leave before it finishes. Cost is what you pay; **latency is what the user feels**, and it is the difference between a product that feels alive and one that feels broken. The single idea that reorganizes everything here is this: a user does not feel the *total* time an answer takes. They feel **time-to-first-token** — the moment output starts. A reply that begins in a few hundred milliseconds and streams over four seconds feels fast, because they are reading from the half-second mark; a reply that stalls silently for two seconds and then dumps the whole answer at once feels slow, at the exact same total. So the plan is: **stream first** (the biggest perceived win, for zero change in work), understand **where the time actually goes** — prefill sets the first-token wait, decode sets the rest — and match each fix to its phase; then **do less waiting** by running independent calls in parallel and caching what repeats; and finally **defend the tail**, because LLM latency at the ninety-ninth percentile runs eight to fifteen times the median and a slice of users always lands there.

### 🎯 What you will be able to do after this lesson

- **Build** latency tooling — a TTFT-vs-total decomposer, a streaming perceived-latency model, a prefill/decode profiler, a TTFT cutter, a decode cutter, a parallel-plus-cache model, and a tail-latency defender — as runnable models.

- **Compare** streaming against blocking on perceived latency, prefill against decode on where the time goes, and p50 against p99 on the tail.

- **Operate** a latency budget (an SLO), a hedged request, and a parallel fan-out.

- **Evaluate** a latency change: did it cut the number the user feels (TTFT) or the total, and did it help the phase that was actually the bottleneck?

> 📦 **Info**
>
> ✅ Before you startIn **13.3** a route to a smaller model was cheaper; it is also *faster* (a small model starts in well under a second), so latency is the second reason to route down-tier. In **13.2** you cut output tokens to save money; the exact same cut cuts latency, because every output token is decode time. And in **12.2** timeouts, retries, and a fallback path defended reliability — the same tools defend the latency tail here. Prompt caching’s mechanics are **12.4**; self-hosted serving is **13.5 / 13.6**.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🍽️ **Analogy**
>
> Latency is **a restaurant kitchen**. Two places take the same forty minutes to cook your meal. The first seats you, brings bread and a drink in two minutes, and the food arrives course by course — you barely notice the wait. The second leaves you at an empty table in silence and then drops everything at once at minute forty — the wait feels endless, even though the total is identical. The bread is **streaming**; the silent table is **blocking**. And a good kitchen also plans for the *worst* table, not the average one — the one order that takes an hour is what gets remembered (the tail). **Where it breaks down:** a kitchen can only cook so fast, while with an LLM you also have real levers on the actual cooking time — which is the middle of this lesson.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“Lower latency means the model computed faster.”** The biggest win, streaming, changes the *felt* wait, not the total work — the model does exactly the same computation.
> - **“The average latency is what matters.”** The tail is what a slice of users always hits, and on LLMs the ninety-ninth percentile runs many times the median — a fast median hides a slow tail.

> 💡 **Info**
>
> ⚠️ Anti-patternTwo habits that waste your effort. **Shipping without streaming** — you leave the single biggest perceived-latency win on the table, optimizing total time while the user still stares at a spinner. And **applying a fix to the wrong phase** — speculative decoding does nothing for a giant RAG prefill, and prompt caching does nothing for a long answer; you have to know which phase is the bottleneck before you pick a lever.

---

## 🎯 Concept 1: Latency Has Two Numbers, And Users Feel TTFT

### Latency Has Two Numbers, And Users Feel TTFT

A request’s latency is time-to-first-token plus decode time, but the user feels only the first-token wait - when output starts. Measure both p50 and p99: LLM tails run eight to fifteen times the median, so a fast median hides a slow tail.

Start with what latency even is. A request’s latency has two parts: **time-to-first-token (TTFT)**, the wait before anything appears, and the **decode time**, the number of output tokens times the time per token, which fills in the rest of the answer. Their sum is the **total**. (In practice that first-token wait also carries a fixed overhead you control — the network round trip to the provider region, the connection setup per call, and any provider-side queue wait — on top of the model’s own prefill; this lesson models the prefill part, but that network slice is real and worth trimming with a nearer region and a warm connection.) But here is the reframe that drives the whole lesson: a person waiting on a screen does not feel the total — they feel **TTFT**, the moment output starts, because after that they are reading. Two replies with the same total feel completely different depending on when the first token lands. And there is a second number you must always watch: the **tail**. LLM APIs have a p50-to-p99 ratio of roughly **one to eight through one to fifteen** (a normal web API is one to two), so a snappy median can hide a p99 of tens of seconds that a real fraction of your users hit every day. The rule from the start: log **TTFT and total separately**, at **p50 and p99**. The block decomposes a request, keyless (illustrative timings).

> 🍞 **Analogy**
>
> It is **bread while the main course cooks**. The kitchen still takes its forty minutes, but if the bread and a drink arrive in the first two, you settle in and the wait evaporates; if the table sits empty until everything lands at once, the same forty minutes feels punishing. TTFT is when the bread arrives. The user is judging the wait by that first thing on the table, not by when the entree is plated — which is why the moment output *starts* matters far more than when it finishes.

Reply A shows its first word almost immediately and streams for four seconds. Reply B shows nothing for two seconds, then prints the whole answer at once. Both finish at about the same total time. Which feels faster?

**📝 `01_two_numbers.py`** - *runnable*

In [ ]:
# Illustrative latency model (ms). PREFILL sets TTFT (grows with INPUT); DECODE sets the rest (per OUTPUT token).
def ttft(in_tok, per_k=200, base=150):     # time-to-first-token = prefill (compute-bound)
    return base + in_tok / 1000 * per_k
TPOT = 20                                  # time-per-output-token = decode (memory-bound), ms/token
def total(in_tok, out_tok, per_k=200, tpot=TPOT):
    return ttft(in_tok, per_k) + out_tok * tpot
# Latency has two numbers, and the user only FEELS one of them: TTFT (when output starts).
in_tok, out_tok = 1000, 200
t_ttft = ttft(in_tok); t_decode = out_tok * TPOT; t_total = total(in_tok, out_tok)
print("One request (in {} / out {} tokens):".format(in_tok, out_tok))
print("  TTFT (first token):  {:.0f} ms   <- what the user FEELS".format(t_ttft))
print("  decode (rest):       {:.0f} ms   ({} tokens x {} ms)".format(t_decode, out_tok, TPOT))
print("  total:               {:.0f} ms".format(t_total))
print()
print("With streaming the user starts reading at {:.0f} ms; without it they stare at a spinner for {:.0f} ms.".format(t_ttft, t_total))
print("Same total work, but the felt wait is {:.1f}x shorter when you stream.".format(t_total / t_ttft))
print()
# The tail is the other half of the story: LLM p50/p99 runs 1:8 to 1:15, not 1:2 like a REST API.
p50_ttft, p99_ttft = 350, 2800
print("And measure BOTH ends: p50 TTFT {} ms vs p99 TTFT {} ms = {:.0f}x.".format(p50_ttft, p99_ttft, p99_ttft / p50_ttft))
print("A fast median hides a slow tail. Log TTFT and total separately, at p50 AND p99.")

# Output:
# One request (in 1000 / out 200 tokens):
#   TTFT (first token):  350 ms   <- what the user FEELS
#   decode (rest):       4000 ms   (200 tokens x 20 ms)
#   total:               4350 ms
#
# With streaming the user starts reading at 350 ms; without it they stare at a spinner for 4350 ms.
# Same total work, but the felt wait is 12.4x shorter when you stream.
#
# And measure BOTH ends: p50 TTFT 350 ms vs p99 TTFT 2800 ms = 8x.
# A fast median hides a slow tail. Log TTFT and total separately, at p50 AND p99.

- A request splits into **TTFT** (the first-token wait, what the user feels) and **decode** (output tokens times the per-token time).
- With streaming the felt wait is TTFT; without it the user stares at a spinner for the whole **total** — many times longer for the same work.
- The **tail** is the other half: p99 TTFT runs several times the p50, so a fast median hides slow requests real users hit.
- The rule: measure TTFT and total **separately**, and always at **p50 and p99** — one number is never enough.

#### 💡 What just happened

⚡ What just happened? Latency is TTFT plus decode, but the user feels only TTFT (when output starts), and you must watch both p50 AND p99 because LLM tails run 8-15x the median. Tradeoff / the framing: optimizing the number the user feels (TTFT) is different from optimizing total work. Next: the single biggest move on that felt number - streaming.

- A request timeline: a spinner to TTFT, then a streaming fill to total
- A felt-wait toggle collapses the wait from total to TTFT; a p50-vs-p99 dial shows the long tail

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Stream The Output

### Stream The Output

Streaming is the highest-ROI latency move because it changes the felt wait, not the total work. The user starts reading at the first token, so perceived latency collapses to TTFT for zero change in total time or cost. Stream first, always.

The first and biggest lever costs almost nothing and touches nothing about the model’s work: **stream the output**. A blocking call withholds the entire answer until it is finished, so the user waits the full total staring at a spinner. A streaming call hands each token to the interface *as it is generated*, so the user starts reading at the **first token** and keeps reading as the rest fills in. The perceived wait collapses from the total down to TTFT — often a **ten-fold or larger** drop — for *zero* change in total time, cost, or compute. And once tokens arrive faster than a person reads, the user never actually waits again after that first token; the answer is always a little ahead of their eyes. The number behind that is **inter-token latency** (ITL, or output tokens per second): streaming feels instant only while tokens outpace reading, so log ITL at p50 and p99 the way you log TTFT — if it sags, the stream stutters and the illusion breaks. This is why streaming is the default for every user-facing LLM feature and the first thing to reach for. It does not make the model faster; it makes the wait *disappear*. The block prices the felt wait both ways, keyless.

> 📺 **Analogy**
>
> It is **subtitles appearing as the words are spoken**, versus waiting for the entire speech to finish before you are handed a transcript. The speech takes just as long either way, but reading along in real time, you are never *waiting* — you are following. Streaming is live subtitles for the model: the answer scrolls out at the pace it is produced, and the viewer is engaged from the first word instead of watching a blank screen until the end.

Streaming an answer instead of blocking changes which of these?

**📝 `02_streaming.py`** - *runnable*

In [ ]:
# Illustrative latency model (ms). PREFILL sets TTFT (grows with INPUT); DECODE sets the rest (per OUTPUT token).
def ttft(in_tok, per_k=200, base=150):     # time-to-first-token = prefill (compute-bound)
    return base + in_tok / 1000 * per_k
TPOT = 20                                  # time-per-output-token = decode (memory-bound), ms/token
def total(in_tok, out_tok, per_k=200, tpot=TPOT):
    return ttft(in_tok, per_k) + out_tok * tpot
# Streaming is the single highest-ROI latency move - it changes the FELT wait, not the total work.
in_tok, out_tok = 1000, 200
t_ttft = ttft(in_tok); t_total = total(in_tok, out_tok)
# Blocking: the whole answer is withheld until it is finished. Streaming: tokens appear as generated.
felt_blocking = t_total     # user sees nothing until the end
felt_streaming = t_ttft     # user starts reading at the first token, then reads as it streams
print("Same request, two delivery modes (total compute is identical at {:.0f} ms):".format(t_total))
print("  blocking  -> first thing on screen at {:.0f} ms (the full answer, all at once)".format(felt_blocking))
print("  streaming -> first thing on screen at {:.0f} ms (then it fills in as you read)".format(felt_streaming))
print()
print("Perceived wait drops {:.1f}x ({:.0f} ms -> {:.0f} ms) for ZERO change in total time or cost.".format(
    felt_blocking / felt_streaming, felt_blocking, felt_streaming))
print("Once tokens arrive faster than a person reads (~{} ms/token here), the user never waits again after the first.".format(TPOT))
print("Stream first. It is the cheapest, biggest win in latency - nothing else touches perceived speed like it.")

# Output:
# Same request, two delivery modes (total compute is identical at 4350 ms):
#   blocking  -> first thing on screen at 4350 ms (the full answer, all at once)
#   streaming -> first thing on screen at 350 ms (then it fills in as you read)
#
# Perceived wait drops 12.4x (4350 ms -> 350 ms) for ZERO change in total time or cost.
# Once tokens arrive faster than a person reads (~20 ms/token here), the user never waits again after the first.
# Stream first. It is the cheapest, biggest win in latency - nothing else touches perceived speed like it.

- **Blocking** shows the first thing on screen only at the *total* time — the whole answer, all at once.
- **Streaming** shows the first token at **TTFT**, then fills in as the user reads — the perceived wait drops many-fold.
- The **total compute is identical** — streaming changes the felt wait, not the work or the cost.
- Once tokens arrive faster than a person reads, the user never waits after the first — which is why streaming is the default.

#### 💡 What just happened

⚡ What just happened? Streaming collapses the perceived wait to TTFT for zero change in total time or cost - the cheapest, biggest latency win, and the reason to stream every user-facing reply. Tradeoff: you handle a token stream in the UI instead of one blob, in exchange for the answer feeling instant. Next: where the time actually goes, so you can cut the real total too.

- Side by side: a blocking response (blank until the end) vs a streaming one (words from TTFT)
- A read-along cursor shows the user never waits again after the first token

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Prefill vs Decode: Where The Time Goes

### Prefill vs Decode: Where The Time Goes

Inference has two phases. Prefill processes the whole prompt at once, is compute-bound, builds the KV cache, and sets TTFT - it grows with the input. Decode generates one token at a time, is memory-bound, reuses the cache, and sets the rest - it grows with the output.

To cut the *real* time (not just the felt wait), you have to know where it goes, and inference has two very different phases. **Prefill** is the model reading your prompt: it processes the *whole input at once*, which is **compute-bound** (lots of parallel matrix math), and it builds the **KV cache** — the attention state the rest of the generation reuses. Prefill is what sets **TTFT**, and it grows with the **input** length. **Decode** is the model writing the answer: it generates *one token at a time*, each step reusing the KV cache, which is **memory-bandwidth-bound**. Decode sets the rest of the total, and it grows with the **output** length. This split is the diagnostic that tells you which lever to pull: a **big RAG context blows up TTFT** (that is all prefill), while a **long answer blows up the total** (that is all decode). Reach for a prefill fix or a decode fix depending on which phase is your bottleneck. The block profiles two requests, keyless.

> 📜 **Analogy**
>
> It is **reading the whole question, then writing the answer one word at a time**. When you get a long exam question, you first read all of it — you can skim the whole thing at once, and a longer question takes longer to read (that is prefill, and it scales with the *question*). Then you write your answer word by word, at a steady pace, and a longer answer takes longer to write (that is decode, and it scales with the *answer*). Reading is fast and parallel; writing is a steady grind. The model works the same way: it ingests the prompt in one pass, then grinds out the reply token by token.

Two requests: (A) a 4,000-token RAG context asking for a one-line answer, and (B) a one-line question asking for a 2,000-word essay. Which is bottlenecked on prefill (TTFT), and which on decode?

**📝 `03_prefill_vs_decode.py`** - *runnable*

In [ ]:
# Illustrative latency model (ms). PREFILL sets TTFT (grows with INPUT); DECODE sets the rest (per OUTPUT token).
def ttft(in_tok, per_k=200, base=150):     # time-to-first-token = prefill (compute-bound)
    return base + in_tok / 1000 * per_k
TPOT = 20                                  # time-per-output-token = decode (memory-bound), ms/token
def total(in_tok, out_tok, per_k=200, tpot=TPOT):
    return ttft(in_tok, per_k) + out_tok * tpot
# WHERE the time goes decides WHICH fix helps. Prefill (input) sets TTFT; decode (output) sets the rest.
for label, in_tok, out_tok in [("short chat", 500, 150), ("RAG (big context)", 4000, 150)]:
    p = ttft(in_tok); d = out_tok * TPOT; t = p + d
    print("{:<18} in {:>4} / out {:>3}:  prefill/TTFT {:>4.0f} ms  +  decode {:>4.0f} ms  =  {:>4.0f} ms total".format(
        label, in_tok, out_tok, p, d, t))
print()
print("Prefill processes the WHOLE prompt at once (compute-bound) and builds the KV cache - it grows with INPUT.")
print("Decode generates one token at a time (memory-bound), reusing the KV cache - it grows with OUTPUT.")
print("So: a big RAG context blows up TTFT (prefill), while a long answer blows up total (decode).")
print("Match the fix to the phase - caching and a shorter prompt cut prefill; shorter output and speculation cut decode.")

# Output:
# short chat         in  500 / out 150:  prefill/TTFT  250 ms  +  decode 3000 ms  =  3250 ms total
# RAG (big context)  in 4000 / out 150:  prefill/TTFT  950 ms  +  decode 3000 ms  =  3950 ms total
#
# Prefill processes the WHOLE prompt at once (compute-bound) and builds the KV cache - it grows with INPUT.
# Decode generates one token at a time (memory-bound), reusing the KV cache - it grows with OUTPUT.
# So: a big RAG context blows up TTFT (prefill), while a long answer blows up total (decode).
# Match the fix to the phase - caching and a shorter prompt cut prefill; shorter output and speculation cut decode.

- **Prefill** processes the whole prompt at once and builds the KV cache — it is compute-bound and grows with the **input**, setting TTFT.
- **Decode** generates one token at a time, reusing the cache — it is memory-bound and grows with the **output**, setting the rest.
- A **big RAG context** makes prefill (TTFT) the bottleneck; a **long answer** makes decode (total) the bottleneck.
- The lesson: identify which phase dominates *before* you pick a fix, because each fix only helps one phase.

#### 💡 What just happened

⚡ What just happened? Inference is prefill (compute-bound, grows with input, sets TTFT, builds the KV cache) then decode (memory-bound, grows with output, sets the rest, reuses the cache). Tradeoff: none - this is the diagnostic that tells you whether a TTFT fix or a decode fix is what you need. Next: the levers that cut prefill, and so cut TTFT.

- Drag an input-size slider (grows the prefill/TTFT bar) and an output-size slider (grows the decode bar)
- The KV cache fills during prefill and is reused during decode

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Cut TTFT: Prompt Caching, A Shorter Prompt, A Faster Model

### Cut TTFT: Prompt Caching, A Shorter Prompt, A Faster Model

TTFT is prefill, so cut prefill. Prompt caching reuses the KV cache for a stable prefix, usually the biggest RAG TTFT win. A shorter prompt is less to prefill. A faster model prefills quicker. Caching helps only prefill - it does nothing for decode.

If TTFT is your bottleneck, you are looking at prefill, and there are three levers. First and biggest, **prompt caching**: when a stable system-plus-context prefix repeats across calls, the model reuses its cached KV state instead of re-processing it, so only the fresh part of the prompt is prefilled. For RAG and long-context apps this is usually the **single largest TTFT win**, and you already met its cost side in 12.4. Second, a **shorter prompt**: fewer input tokens is simply less to prefill (this is the 13.2 trimming, now paying a latency dividend). Third, a **faster model**: a smaller model prefills quicker (the 2026 fastest-to-first-token models — Google’s Gemini Flash and Anthropic’s Claude Haiku, with OpenAI’s small tiers close behind — start in well under a second on a medium prompt; that is the 13.3 route, paying a latency dividend). The one thing to keep straight: **all three help prefill only** — prompt caching does *nothing* for the decode phase, so if your problem is a long answer, none of this helps. The block prices the three TTFT levers, keyless.

> 🥣 **Analogy**
>
> It is **a chef with the mise en place already prepped**. Every dish at this restaurant starts from the same base — the stock, the chopped aromatics, the standing prep — and a good kitchen does that work *once*, at the start of service, and reuses it all night. When your order comes in, the chef is not chopping onions from scratch; they start from the ready base and only do the part unique to your dish. Prompt caching is that mise en place: the stable prefix is prepped and reused, so each request only pays to process the fresh part — and the first bite (the first token) comes out far sooner.

Your RAG app sends a 3,500-token system-plus-context prefix that never changes, plus a short fresh query. You turn on prompt caching. What happens to TTFT and to decode?

**📝 `04_cut_ttft.py`** - *runnable*

In [ ]:
# Illustrative latency model (ms). PREFILL sets TTFT (grows with INPUT); DECODE sets the rest (per OUTPUT token).
def ttft(in_tok, per_k=200, base=150):     # time-to-first-token = prefill (compute-bound)
    return base + in_tok / 1000 * per_k
TPOT = 20                                  # time-per-output-token = decode (memory-bound), ms/token
def total(in_tok, out_tok, per_k=200, tpot=TPOT):
    return ttft(in_tok, per_k) + out_tok * tpot
# Cut TTFT = cut PREFILL. Three levers: prompt caching, a shorter prompt, and a faster model.
stable_prefix, fresh = 3500, 500          # a RAG prompt: a stable system+context prefix + a fresh query
cold = ttft(stable_prefix + fresh)        # cold: prefill the whole 4000-token prompt
cached = ttft(fresh) + 30                 # cached: the 3500 prefix is reused, only the 500 fresh tokens prefill
faster_model = ttft(stable_prefix + fresh, per_k=100)   # a faster/smaller model prefills quicker
print("TTFT for a {}-token RAG prompt (stable prefix {} + fresh {}):".format(stable_prefix + fresh, stable_prefix, fresh))
print("  cold (no cache):        {:.0f} ms".format(cold))
print("  prompt caching:         {:.0f} ms   ({:.1f}x faster - the stable prefix is not re-processed)".format(cached, cold / cached))
print("  faster model (prefill): {:.0f} ms   ({:.1f}x faster)".format(faster_model, cold / faster_model))
print()
print("Prompt caching is usually the biggest single TTFT win for RAG and long context (it skips the expensive prefill).")
print("Caching helps ONLY prefill/TTFT - it does nothing for the decode phase. Pick the lever that hits your phase.")

# Output:
# TTFT for a 4000-token RAG prompt (stable prefix 3500 + fresh 500):
#   cold (no cache):        950 ms
#   prompt caching:         280 ms   (3.4x faster - the stable prefix is not re-processed)
#   faster model (prefill): 550 ms   (1.7x faster)
#
# Prompt caching is usually the biggest single TTFT win for RAG and long context (it skips the expensive prefill).
# Caching helps ONLY prefill/TTFT - it does nothing for the decode phase. Pick the lever that hits your phase.

- **Prompt caching** reuses the stable prefix’s prefill, so only the fresh tokens are processed — usually the biggest TTFT drop for RAG.
- A **faster model** prefills each token quicker, cutting TTFT further (that is the 13.3 route paying off in speed).
- A **shorter prompt** is simply fewer input tokens to prefill (the 13.2 trim paying off in speed).
- All three hit **prefill only** — none of them touches decode, so match the lever to the phase.

#### 💡 What just happened

⚡ What just happened? TTFT is prefill, cut with prompt caching (reuse the stable prefix - the biggest RAG win), a shorter prompt, and a faster model - none of which help decode. Tradeoff: caching commits you to a stable prefix layout, in exchange for a much lower first-token wait. Next: the levers for the OTHER phase, decode, which sets the total.

- A RAG prompt as a stable prefix + a fresh query; toggle prompt caching and the prefix greys out as TTFT drops
- Toggle a faster model and per-token prefill shrinks; the decode bar never moves

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Cut Decode: Shorter Output And Speculative Decoding

### Cut Decode: Shorter Output And Speculative Decoding

Total is TTFT plus output times the per-token time, so cut the output or speed up decode. A shorter answer cuts total directly. Speculative decoding has a draft model propose tokens the target verifies in one pass - about two to three times faster decode, with identical output. It cannot cut TTFT.

If your bottleneck is a long answer, you are looking at decode, and total equals TTFT plus output tokens times the per-token time — so cut the output, or make each token faster. First, the **shorter output**: this is the 13.2 move (structured answers, a length budget, a `max_tokens` cap), and every token you do not generate is decode time you do not spend — it cuts total directly (and cost). Second, **speculative decoding**: a small **draft model proposes several tokens**, and the big **target model verifies them all in a single forward pass**, accepting the ones it agrees with. Because verification is parallel, you get roughly **two to three times faster decode** with *identical* output to running the target alone (self-speculative variants skip layers instead of using a separate draft). The catch to keep straight: speculative decoding speeds **decode only** — it *cannot* cut TTFT, because prefill still has to run before the first verified token. (And note: a cheap-model-drafts-then-strong-model-*refines* pattern is a different thing — that is the **model cascade** from 13.3, not token-level speculative decoding.) The block prices the decode levers, keyless.

> ⌨️ **Analogy**
>
> Speculative decoding is **a fast typist drafting while the expert just checks**. Instead of the slow expert composing every word from scratch, a quick junior types out the next sentence as a guess, and the expert scans the whole sentence at once and keeps the parts that are right — far faster than writing it themselves, and the final text is exactly what the expert would have produced, because they verified every word. The draft model is the fast typist; the target model is the expert who signs off. You get the expert’s answer at closer to the typist’s speed.

Speculative decoding gives you a two-to-three-times faster generation. On a request with a huge RAG context but a short answer, how much does it help?

**📝 `05_cut_decode.py`** - *runnable*

In [ ]:
# Illustrative latency model (ms). PREFILL sets TTFT (grows with INPUT); DECODE sets the rest (per OUTPUT token).
def ttft(in_tok, per_k=200, base=150):     # time-to-first-token = prefill (compute-bound)
    return base + in_tok / 1000 * per_k
TPOT = 20                                  # time-per-output-token = decode (memory-bound), ms/token
def total(in_tok, out_tok, per_k=200, tpot=TPOT):
    return ttft(in_tok, per_k) + out_tok * tpot
# Cut total = cut DECODE. total = TTFT + out x TPOT, so cut the output tokens, or speed up decode.
in_tok = 1000; t_ttft = ttft(in_tok)
verbose, short = 200, 80
t_verbose = t_ttft + verbose * TPOT
t_short = t_ttft + short * TPOT
# Speculative decoding: a small draft model proposes tokens, the target verifies them in one pass.
# ~2.5 tokens accepted per step -> effective decode ~2.5x faster. It speeds DECODE, not prefill/TTFT.
tpot_spec = TPOT / 2.5
t_spec = t_ttft + verbose * tpot_spec
print("total = TTFT ({:.0f} ms) + output x {} ms/token:".format(t_ttft, TPOT))
print("  verbose answer ({} tok):        {:.0f} ms".format(verbose, t_verbose))
print("  structured/short answer ({} tok): {:.0f} ms   ({:.1f}x faster - fewer tokens to decode)".format(short, t_short, t_verbose / t_short))
print("  speculative decoding ({} tok):   {:.0f} ms   ({:.1f}x faster decode; TTFT unchanged at {:.0f} ms)".format(
    verbose, t_spec, t_verbose / t_spec, t_ttft))
print()
print("Shorter output cuts total directly (and cost - Lesson 13.2). Speculation cuts the per-token decode time.")
print("Speculative decoding helps DECODE only - it cannot cut TTFT, because prefill still has to run first.")

# Output:
# total = TTFT (350 ms) + output x 20 ms/token:
#   verbose answer (200 tok):        4350 ms
#   structured/short answer (80 tok): 1950 ms   (2.2x faster - fewer tokens to decode)
#   speculative decoding (200 tok):   1950 ms   (2.2x faster decode; TTFT unchanged at 350 ms)
#
# Shorter output cuts total directly (and cost - Lesson 13.2). Speculation cuts the per-token decode time.
# Speculative decoding helps DECODE only - it cannot cut TTFT, because prefill still has to run first.

- **Total** is TTFT plus output tokens times the per-token decode time — so fewer output tokens means a lower total directly.
- A **structured / shorter answer** cuts the total (and the cost — the 13.2 lever paying off in speed).
- **Speculative decoding** speeds the per-token decode (about two to three times) with identical output — but **TTFT is unchanged**.
- Speculation helps **decode only**; on a big-context, short-answer request the bottleneck is prefill, so it barely helps.

#### 💡 What just happened

⚡ What just happened? Decode sets the total; cut it with a shorter output (13.2, also cheaper) and speculative decoding (a draft proposes, the target verifies in one pass, ~2-3x decode, identical output) - neither of which cuts TTFT. Tradeoff: speculation is a serving-side feature you enable, not free everywhere. Next: removing waiting entirely with parallelism and caches.

- An output-length slider shrinks the total; a speculative-decoding toggle halves the per-token decode
- TTFT stays put while the decode bar shrinks - speculation is decode-only

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Do Less Waiting: Parallel Calls And Semantic Caching

### Do Less Waiting: Parallel Calls And Semantic Caching

If a request makes independent calls, run them concurrently so wall-clock is the max, not the sum - N calls in the time of the slowest. And a semantic cache returns a stored answer in milliseconds for a repeated or near-identical question, skipping the model entirely.

The fastest work is the work you avoid. Two techniques remove waiting outright. First, **parallelize independent calls**: if a single request makes several model calls that do *not* depend on each other — embed a query, classify intent, and fetch a summary, say — running them one after another pays the *sum* of their times, but running them concurrently (with `asyncio` or threads) makes the wall-clock the **max**, the slowest single call. Four independent calls finish in the time of one. (Calls that *depend* on each other cannot be parallelized — you need the first result to make the second call.) Second, **semantic caching**: a repeated or near-identical question returns a **stored answer in milliseconds** instead of seconds, matched at the *intent* level by embedding similarity (not exact string), so “what are your hours?” and “when are you open?” hit the same entry. On a repetitive workload this can cut mean latency dramatically, because a cache hit skips the model entirely. The block prices both, keyless.

> 🏃 **Analogy**
>
> It is **sending four errands to four people at once**, instead of doing them yourself one after another. If you run to the bank, the post office, the pharmacy, and the grocer in sequence, you pay the sum of all four trips; hand each errand to a different friend and they all go at once, so you are done when the *slowest* friend gets back. Parallel calls are those four friends. And the semantic cache is the friend who says “I already picked that up yesterday, here it is” — the errand you do not have to run at all.

A request makes three model calls: (1) classify the query, (2) using that class, fetch a template, (3) fill the template. Can all three run in parallel?

**📝 `06_parallel_and_cache.py`** - *runnable*

In [ ]:
# Illustrative latency model (ms). PREFILL sets TTFT (grows with INPUT); DECODE sets the rest (per OUTPUT token).
def ttft(in_tok, per_k=200, base=150):     # time-to-first-token = prefill (compute-bound)
    return base + in_tok / 1000 * per_k
TPOT = 20                                  # time-per-output-token = decode (memory-bound), ms/token
def total(in_tok, out_tok, per_k=200, tpot=TPOT):
    return ttft(in_tok, per_k) + out_tok * tpot
# Do less waiting: run INDEPENDENT calls in parallel (wall-clock = max, not sum), and cache repeats.
per_call = 1200; n = 4
serial = n * per_call        # four calls one after another
parallel = per_call          # four independent calls at once -> as slow as the slowest one
print("Four INDEPENDENT model calls at {} ms each:".format(per_call))
print("  serial (one after another): {} ms".format(serial))
print("  parallel (all at once):     {} ms   ({}x faster - wall-clock is the MAX, not the sum)".format(parallel, serial // parallel))
print()
# Semantic caching: a repeated or near-identical question returns a stored answer in milliseconds.
hit_rate, cache_ms = 0.73, 5      # illustrative high-repetition workload
mean = hit_rate * cache_ms + (1 - hit_rate) * per_call
print("Semantic cache on a repetitive workload (hit rate {:.0f}%, a hit returns in {} ms):".format(hit_rate * 100, cache_ms))
print("  mean latency: {} ms -> {:.0f} ms   ({:.1f}x faster; a cache hit skips the model entirely)".format(per_call, mean, per_call / mean))
print("Parallelize what is independent; cache what repeats. The fastest call is the one you never make.")

# Output:
# Four INDEPENDENT model calls at 1200 ms each:
#   serial (one after another): 4800 ms
#   parallel (all at once):     1200 ms   (4x faster - wall-clock is the MAX, not the sum)
#
# Semantic cache on a repetitive workload (hit rate 73%, a hit returns in 5 ms):
#   mean latency: 1200 ms -> 328 ms   (3.7x faster; a cache hit skips the model entirely)
# Parallelize what is independent; cache what repeats. The fastest call is the one you never make.

- Four **independent** calls run **serially** pay the sum of their times; run **in parallel** they finish in the time of the slowest — wall-clock is the max, not the sum.
- Only **independent** calls parallelize; a dependency chain (each call needs the last result) has to run in order.
- A **semantic cache** serves a repeat in milliseconds, matched by intent, not exact string — a hit skips the model entirely.
- On a repetitive workload the cache drops mean latency sharply, because most requests never reach the model.

#### 💡 What just happened

⚡ What just happened? Remove waiting: run independent calls in parallel (wall-clock = max, not sum) and serve repeats from a semantic cache (milliseconds on a hit, matched by intent). Tradeoff: parallelism needs genuinely independent calls, and a semantic cache risks a wrong hit if the similarity threshold is too loose. Next: the requests that are still slow no matter what - the tail.

- Four independent calls stacked serially, then snapping into parallel (wall-clock = the slowest)
- A semantic-cache hit-rate slider drops mean latency as repeats return in milliseconds

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Defend The Tail

### Defend The Tail

The tail is what a slice of users always hits, and LLM p99 runs eight to fifteen times the median. Set a latency budget (an SLO), measure p99, and defend it with a hard timeout, a hedged request (fire a second attempt after p50, take the first), and a fast fallback.

Everything so far optimized the *median*. But a real fraction of your users land on the **tail** — the slowest requests — and on LLMs the tail is brutal: p99 runs **eight to fifteen times** the median, so a fast median hides a p99 of tens of seconds. You cannot ignore it, because it is not noise; it is a predictable slice of every day’s traffic. So you **set a latency budget** as an SLO (the 2026 interactive benchmarks target a few hundred milliseconds of TTFT at p99), you **measure p50 and p99** for TTFT and total, and you **defend the tail** with the reliability tools from 12.2: a **hard timeout** at the budget so a stuck request cannot hang forever; a **hedged request** — fire a second attempt once the first passes a chosen percentile, and take whichever answers first, which bounds the tail near twice that percentile instead of the raw p99. The percentile you fire at is a knob: firing late (around the p95) hedges only the genuinely slow requests and barely adds load — the load-safe default from *The Tail at Scale* — while firing early (at the p50, as the block does to show the tightest bound) is aggressive and can nearly double your traffic. And a **fast fallback model** for when even that is slow. Load makes the tail worse (queuing grows the p99 as utilization climbs), so headroom helps too. The whole discipline: **optimize the median with streaming and caching; defend the tail with timeouts, hedging, and a fallback**. The block prices the tail defense, keyless.

> 🛒 **Analogy**
>
> It is **staffing for the slowest checkout lane, not the average one**. Most shoppers get through in a couple of minutes, but the one customer with a price check and a coupon dispute holds up the line for fifteen — and that is the trip people remember and complain about. A good store does not just measure the average wait; it watches the *worst* lane and puts a floating cashier on standby to open a second register the moment a line backs up. The hedged request is that standby cashier: when one attempt is taking too long, you open a second lane and serve whoever is ready first.

Your p50 latency is excellent, but a slice of users complains the app is unbearably slow. What is going on, and what is the fix?

**📝 `07_defend_the_tail.py`** - *runnable*

In [ ]:
# Illustrative latency model (ms). PREFILL sets TTFT (grows with INPUT); DECODE sets the rest (per OUTPUT token).
def ttft(in_tok, per_k=200, base=150):     # time-to-first-token = prefill (compute-bound)
    return base + in_tok / 1000 * per_k
TPOT = 20                                  # time-per-output-token = decode (memory-bound), ms/token
def total(in_tok, out_tok, per_k=200, tpot=TPOT):
    return ttft(in_tok, per_k) + out_tok * tpot
# The tail is what a slice of users ALWAYS hit. Set a budget, measure p99, and defend it.
p50, p99 = 400, 4000              # TTFT ms; LLM p50/p99 runs 1:8 to 1:15 (not 1:2 like a REST API)
BUDGET_P99 = 800                  # an SLO: 800 ms TTFT at p99
print("TTFT distribution: p50 {} ms, p99 {} ms  = {:.0f}x (LLM tails are long).".format(p50, p99, p99 / p50))
print("SLO: {} ms TTFT at p99 -> the raw p99 of {} ms BREACHES it by {:.1f}x.".format(BUDGET_P99, p99, p99 / BUDGET_P99))
print()
# Defend the tail: a hard timeout, a HEDGED request (fire a second attempt after p50, take the first to answer),
# and a fast fallback model. Hedging bounds the tail near ~2x p50 instead of the raw p99.
p99_hedged = 2 * p50 + 100        # fire the hedge at p50, the faster of the two usually wins
print("Defend it: timeout at the budget + a HEDGED request (fire a 2nd attempt at p50, take the first) + a fast fallback.")
print("  p99 with hedging: {} ms -> {} ms   (back near the {} ms budget)".format(p99, p99_hedged, BUDGET_P99))
print("Optimize the median with streaming and caching; defend the tail with timeouts, hedging, and a fallback (Lesson 12.2).")

# Output:
# TTFT distribution: p50 400 ms, p99 4000 ms  = 10x (LLM tails are long).
# SLO: 800 ms TTFT at p99 -> the raw p99 of 4000 ms BREACHES it by 5.0x.
#
# Defend it: timeout at the budget + a HEDGED request (fire a 2nd attempt at p50, take the first) + a fast fallback.
#   p99 with hedging: 4000 ms -> 900 ms   (back near the 800 ms budget)
# Optimize the median with streaming and caching; defend the tail with timeouts, hedging, and a fallback (Lesson 12.2).

**📝 `latency-stack.py`** - *illustrative*

```python
# LATENCY STACK - an illustrative minimal example (stream + cached prefix + capped output + a tail guard).
import anthropic

client = anthropic.Anthropic()
CACHED_SYSTEM = SYSTEM_PROMPT + FEW_SHOT             # stable across calls -> cache it so prefill (TTFT) stays low

def answer(prompt):
    # stream so the user reads from the FIRST token; cap output so decode stays short
    with client.messages.stream(
        model="claude-haiku-4-5", max_tokens=256,               # a fast model + a short answer
        system=[{"type": "text", "text": CACHED_SYSTEM,
                 "cache_control": {"type": "ephemeral"}}],       # cached prefix -> low TTFT (Lesson 12.4)
        messages=[{"role": "user", "content": prompt}],
    ) as stream:
        ttft = None
        for text in stream.text_stream:
            if ttft is None:
                ttft = "mark first-token time here"              # log TTFT SEPARATELY from total (Lesson 12.8)
            yield text                                           # hand each token to the UI as it arrives
# Defend the tail (Lesson 12.2): wrap this in a timeout at your p99 budget, hedge a 2nd attempt after p50,
# and fall back to another fast model. And meter TTFT and total at p50 AND p99 - or you are optimizing blind.
# Output: (illustrative minimal example - needs anthropic + a key + a running UI; not run here.)
```

- The **tail** is a predictable slice of traffic, and LLM p99 runs many times the p50 — so a raw p99 easily **breaches** a latency budget.
- A **hedged request** (fire a second attempt at p50, take the first to answer) bounds the tail near twice the median, pulling p99 back near the budget.
- You pair it with a **hard timeout** at the budget and a **fast fallback** model — the reliability tools from 12.2, applied to latency.
- The **illustrative stack** ties it together: stream, cache the stable prefix, cap the output, run a fast model, and guard the tail — and meter TTFT and total at p50 and p99.

#### 💡 What just happened

⚡ What just happened? The tail is a predictable slice and LLM p99 runs 8-15x the median, so set a budget, measure p99, and defend it with a timeout, a hedged request (fire a 2nd at p50, take the first), and a fast fallback. Tradeoff: a hedge doubles the work on the slow slice, in exchange for a bounded tail. That is the whole lesson: optimize the median, defend the tail, measured.

- A latency histogram with a p99 marker way past a budget line
- Fire a hedged request at p50 and a timeout at the budget, and the tail collapses back inside the budget

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> Latency optimization is a stack, and each layer targets a different thing. It starts from the reframe that **the user feels TTFT, not total**, and that you must watch **p50 and p99** (Step 1). The first and biggest move is to **stream**, which collapses the felt wait to TTFT for zero extra work (Step 2). To cut the real time you find **where it goes** — prefill sets TTFT, decode sets the rest (Step 3) — and match the fix to the phase: **caching, a shorter prompt, and a faster model** cut prefill/TTFT (Step 4), while a **shorter output and speculative decoding** cut decode/total (Step 5). Then you **remove waiting** with parallel calls and a semantic cache (Step 6). And you **defend the tail** with a budget, a timeout, a hedged request, and a fallback (Step 7). Ask two questions of any latency change: did it cut the number the user **feels** (TTFT), and did it hit the phase that was actually the **bottleneck**?

| Technique | Phase it targets | When to reach for it |
|---|---|---|
| Streaming | perceived (TTFT) | always, every user-facing reply |
| Prompt caching | prefill / TTFT | a stable prefix (RAG, long context) |
| Faster model / shorter prompt | prefill / TTFT | the first-token wait is the bottleneck |
| Shorter output | decode / total | the answer is longer than it needs to be |
| Speculative decoding | decode / total | long generations on a serving stack that supports it |
| Parallel calls + semantic cache | whole request | independent calls, or repeated queries |
| Timeout + hedge + fallback | the tail (p99) | defending an SLO against slow requests |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now make an AI feature feel instant and keep its tail in check. Controlling the serving layer itself — continuous batching, KV-cache quantization, and speculative decoding on your own GPU — comes later in this module, when you self-host in Lesson 13.5 and Lesson 13.6. Seeing your TTFT and total at p50 and p99 in production is the observability of Lesson 12.8, and the timeout-hedge-fallback tail defense builds on the reliability of Lesson 12.2. The primary references are [prefill vs decode](https://redis.io/blog/prefill-vs-decode/), [speculative decoding](https://arxiv.org/abs/2211.17192), [Anthropic streaming](https://docs.anthropic.com/en/docs/build-with-claude/streaming) and [prompt caching](https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching), and [The Tail at Scale](https://research.google/pubs/the-tail-at-scale/) on hedged requests.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **LatencyOptimization**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-13_4.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-13.4.html` - regenerate this notebook whenever the source HTML is updated.*
